# Partial Payments, D365 Journals, and Reporting

This workflow uses three finance agents: **Partial Payment Agent**, **D365 Journal Agent**, and **Reporting Agent**. It defaults to approval-safe journal preparation; it never contacts a real D365 tenant.

In [1]:
from dataclasses import asdict
import pandas as pd
from project_02_partial_payment_erp_reporting_agents import FinanceOperationsOrchestrator, demo_data

payments, invoices = demo_data()
workflow = FinanceOperationsOrchestrator()
allocations, postings, report = workflow.run(payments, invoices, approve_posting=False)
print(f'Processed {len(allocations)} receipts; all journals remain approval-safe.')

Processed 3 receipts; all journals remain approval-safe.


## Partial-payment allocation

The agent applies the lesser of receipt amount and open balance. Residuals remain open; overpayments become customer credit.

In [2]:
allocation_table = pd.DataFrame([asdict(item) for item in allocations])
allocation_table

,payment_id,invoice_id,applied_amount,remaining_invoice_balance,variance_amount,action,note
0,RCPT-2001,INV-2001,600.0,400.0,400.0,keep invoice open,Partial payment applied; residual stays open f...
1,RCPT-2002,INV-2002,750.0,0.0,0.0,close invoice,Invoice is fully settled.
2,RCPT-2003,INV-2003,500.0,0.0,50.0,create customer credit,Overpayment retained as customer credit.


## D365 Finance journal preview

Each voucher must balance before it can move to approval or a production D365 API adapter.

In [3]:
journal_lines = [asdict(entry) for posting in postings for entry in posting.entries]
journal_table = pd.DataFrame(journal_lines)
display(journal_table)
checks = journal_table.groupby('voucher')[['debit', 'credit']].sum()
display(checks)
assert (checks.debit == checks.credit).all()
print('✓ Every prepared voucher is balanced.')

,voucher,account,debit,credit,currency,description,dimension
0,PAY-RCPT-2001,110100,600.0,0.0,USD,Bank receipt RCPT-2001,BUSINESSUNIT-001
1,PAY-RCPT-2001,130100,0.0,600.0,USD,Settle INV-2001,BUSINESSUNIT-001
2,PAY-RCPT-2002,110100,750.0,0.0,USD,Bank receipt RCPT-2002,BUSINESSUNIT-001
3,PAY-RCPT-2002,130100,0.0,750.0,USD,Settle INV-2002,BUSINESSUNIT-001
4,PAY-RCPT-2003,110100,550.0,0.0,USD,Bank receipt RCPT-2003,BUSINESSUNIT-001
5,PAY-RCPT-2003,130100,0.0,500.0,USD,Settle INV-2003,BUSINESSUNIT-001
6,PAY-RCPT-2003,210250,0.0,50.0,USD,Customer credit from RCPT-2003,BUSINESSUNIT-001


,debit,credit
voucher,,
PAY-RCPT-2001,600.0,600.0
PAY-RCPT-2002,750.0,750.0
PAY-RCPT-2003,550.0,550.0


✓ Every prepared voucher is balanced.


## Reconciliation report

The reporting agent produces operational measures and keeps posting exceptions visible.

In [5]:
report_table = pd.DataFrame([report])
display(report_table)
assert report['partial_payments'] == 1
assert report['customer_credit_created'] == 50.0
assert all(posting.status == 'pending approval' for posting in postings)
print('✓ Reporting and approval controls passed.')

,generated_at_utc,payments_processed,amount_applied,remaining_open_balance,customer_credit_created,partial_payments,journals_ready_or_posted,exceptions
0,2026-08-06T06:53:08+00:00,3,1850.0,400.0,50.0,1,3,[]


✓ Reporting and approval controls passed.
